# M1'' — KG Single-Absorber Suppression vs a **Footprint-Matched Gated-Dense** Control

**Artifact:** iter-7 decisive control + honest forget-operating-point test for the auditability-first
**Counterfactual Co-Response Grouping (CCRG)** units (cluster-level SAE features).

**The claim under test.** Ablating *one* KG-named SAE **absorber latent** (**KG-ABL**) is a better
*unlearning handle* than a strong dense baseline. iter-6 compared KG-ABL against a sub-context-labeled
diff-of-means `u_sub` — but that was **unfair on token footprint**: KG-ABL edits only the ~1–3 % of tokens
where its sparse feature fires, while ungated `u_sub` edits **every** token. iter-7 adds the **fair**
control **DENSE-SUB-ABL-GATED**: erase `u_sub` only where it fires above a threshold τ, with τ calibrated so
the gate's firing footprint equals the absorber's footprint `f_kg`.

**What this notebook reproduces (no GPU, no LLM calls).** The full experiment runs Gemma-2-2b under SAE
edit hooks and scores 2109 generations with two LLM judges. That heavy step is already done; its
**per-prompt judged generations** are released as data. Here we recompute, from that data, the *decisive
statistic* of the paper:

1. the **joint retain-utility** of each operator = `harmonic_mean(fluency, content_preservation)`;
2. the **paired bootstrap** confidence interval of the decisive advantage `KG-ABL − GATED-DENSE`;
3. the per-case **3-way fork** verdict; and
4. the **aggregate** verdict (`absorption_exceeds_cofiring`, overall verdict).

The bootstrap and harmonic-mean code are copied **verbatim** from `core.py` / `method.py`. The
GPU-derived `meaningful_forget` flags (completion-accuracy + frozen sub-probe drops) and the second-judge
flags are loaded precomputed.

**Headline:** for the *concentrated* spelling absorber `large`, KG-ABL beats the footprint-matched gated
dense by **+1.58** joint (CI excludes 0) → `KG_BEATS_GATED_DENSE`; for *distributed* country senses
(Georgia) the single latent cannot induce real forgetting → `NO_MEANINGFUL_FORGET`. Overall:
**`SPARSE_SAE_HANDLE_ESTABLISHED`**, but feature-dependent.

In [ ]:
# --- Install dependencies (Colab-safe) ---
# The demo only needs numpy (bootstrap) + matplotlib (plot). Both are pre-installed on Colab,
# so they go behind the google.colab guard (installing them ON Colab corrupts the C extensions).
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages: pre-installed on Colab; install locally at Colab's exact versions to match.
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

In [ ]:
# --- Imports (stdlib + numpy from the original; matplotlib added for the demo plot) ---
import os, json, math
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# --- Data loading: GitHub raw URL with local fallback (Colab-compatible) ---
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7ee30c-catching-silent-feature-absorption-in-fr/main/round-7/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("dataset:", data["dataset"])
print("judge   :", data["judge"])
print("n per-prompt examples:", len(data["examples"]))
print("cases in this mini subset:", list(data["per_case_meta"].keys()))

## Config

All tunable parameters live here. The analysis is a pure-NumPy bootstrap over ~100 judged prompts, so the
**original full values run in well under a second** — no scaling-down is needed. `B_BOOT` is the number of
bootstrap resamples; reduce it for a faster (noisier) CI.

In [ ]:
# ---- Tunable config (these are the ORIGINAL full-run values; the demo runs them as-is) ----
SEED   = data.get("seed", 1234)       # RNG seed for the paired bootstrap (original: 1234)
B_BOOT = data.get("B_boot", 10000)    # bootstrap resamples (original: 10000; runs <1s). Try 2000 for speed.

# Preservation roles over which the joint retain-utility is measured (original: ("RETAIN","UNRELATED"))
PRES = ("RETAIN", "UNRELATED")

# 3-way fork verdict labels (verbatim from method.py)
F_BEATS  = "KG_BEATS_GATED_DENSE_AT_MEANINGFUL_FORGET"
F_CLOSES = "GATED_DENSE_CLOSES_GAP_DISCOVERY_IS_THE_VALUE"
F_NOMEAN = "NO_MEANINGFUL_FORGET_SCOPE_TO_PARTIAL_SUPPRESSION"

print(f"SEED={SEED}  B_BOOT={B_BOOT}  PRES={PRES}")

## Core statistics (verbatim from the experiment code)

Two functions carry the whole analysis, copied unchanged from the artifact:

- **`harmonic_mean(fluency, content_pres)`** — the per-prompt *joint retain-utility*. A generation is only
  good if it is *both* fluent **and** preserves unrelated content; the harmonic mean punishes either failing.
- **`paired_bootstrap_diff(a, b)`** — a paired bootstrap (B resamples) of the mean difference `a − b`, with
  a 95 % CI and an `excl_0` flag (does the CI exclude zero?). Pairing is per-prompt, so it controls for
  prompt difficulty.

In [ ]:
# --- verbatim from method.py / core.py ---
rng = np.random.default_rng(SEED)

def harmonic_mean(f, c):
    f = float(f); c = float(c)
    if f <= 0 and c <= 0:
        return 0.0
    return (2.0 * f * c) / (f + c + 1e-9)

def paired_bootstrap_diff(a, b, B=B_BOOT):
    a = np.asarray(a, float); b = np.asarray(b, float)
    n = len(a)
    if n == 0:
        return {"diff": 0.0, "ci_lo": 0.0, "ci_hi": 0.0, "excl_0": False, "n": 0}
    idx = rng.integers(0, n, size=(B, n))
    d = a[idx].mean(1) - b[idx].mean(1)
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {"diff": float(a.mean() - b.mean()), "ci_lo": float(lo), "ci_hi": float(hi),
            "excl_0": bool(lo > 0 or hi < 0), "n": int(n)}

## Per-case decisive statistic + 3-way fork

For each case we pull the judged preservation prompts (roles `RETAIN`/`UNRELATED`), build the per-prompt
joint utility for **KG-ABL** and for **DENSE-SUB-ABL-GATED**, and bootstrap their paired difference. The
sign convention is `diff > 0` ⇒ **KG wins** (higher retained utility at matched forget).

`paired_joint` mirrors `_paired_util` and `fork_verdict` mirrors `_joint_and_fork` from `method.py`. The
fork needs two GPU-derived inputs that can't be recomputed without the model — `kg_can_forget` /
`gated_can_forget` (the `meaningful_forget` proof) and the second-judge agreement — so those are loaded
from `per_case_meta`. Everything else (the decisive CI, the advantage, the verdict) is recomputed here.

In [ ]:
examples       = data["examples"]
per_case_meta  = data["per_case_meta"]

def paired_joint(rows):
    """Per-prompt joint utility = harmonic_mean(fluency, content_pres) for KG-ABL vs GATED.
    Mirrors method.py _paired_util over the preservation prompts (skip any unjudged prompt)."""
    uK, uG = [], []
    for e in rows:
        fk, ck = e["metadata_fluency_kg"],    e["metadata_content_pres_kg"]
        fg, cg = e["metadata_fluency_gated"], e["metadata_content_pres_gated"]
        if None in (fk, ck, fg, cg):
            continue
        uK.append(harmonic_mean(fk, ck))
        uG.append(harmonic_mean(fg, cg))
    return np.array(uK), np.array(uG)

def fork_verdict(kg_can, gated_can, ci, second_favors_kg, second_available):
    """3-way fork — verbatim logic from method.py _joint_and_fork."""
    meaningful   = kg_can and gated_can
    p_excl0      = ci["excl_0"]
    adv          = ci["diff"]
    p_favors_kg  = p_excl0 and adv > 0
    if not kg_can:
        return F_NOMEAN
    elif p_excl0 and p_favors_kg and meaningful and (second_favors_kg if second_available else True):
        return F_BEATS
    elif not p_excl0:
        return F_CLOSES
    else:
        return F_CLOSES

cases = {}
for cid, m in per_case_meta.items():
    rows = [e for e in examples if e["metadata_case"] == cid and e["metadata_role"] in PRES]
    uK, uG = paired_joint(rows)
    ci = paired_bootstrap_diff(uK, uG)
    fork = fork_verdict(m["kg_can_forget"], m["gated_can_forget"], ci,
                        m["second_judge_favors_kg"], m["second_judge_available"])
    cases[cid] = {"regime": m["regime"], "absorber": m["absorber_latent"],
                  "underpowered": m["u_sub_underpowered"],
                  "kg_can_forget": m["kg_can_forget"], "gated_can_forget": m["gated_can_forget"],
                  "ci": ci, "adv": ci["diff"], "fork": fork,
                  "kg_util": float(uK.mean()) if len(uK) else None,
                  "gated_util": float(uG.mean()) if len(uG) else None,
                  "published_fork": m["published_fork_verdict"],
                  "published_adv": m["published_adv_KG_vs_GATED"]}
    print(f"{cid:18s} | regime={m['regime']:10s} | n={ci['n']:3d}")
    print(f"   adv (KG - GATED) = {ci['diff']:+.3f}   95% CI [{ci['ci_lo']:+.3f}, {ci['ci_hi']:+.3f}]   excl_0={ci['excl_0']}")
    print(f"   kg_can_forget={m['kg_can_forget']}  ->  FORK = {fork}")
    print(f"   (published fork = {m['published_fork_verdict']}, published adv = {m['published_adv_KG_vs_GATED']:+.3f})\n")

## Why KG wins on `large`: a qualitative look

The win is not subtle. On preservation prompts (which have *nothing* to do with the word "large"), erasing
the dense `u_sub` direction makes the model **collapse into repeating "large"** and lose the content —
content-preservation drops to 0. Ablating the *single* absorber latent 8463 (KG-ABL) leaves the
continuation intact. This is exactly what the joint-utility advantage measures.

In [ ]:
large_rows = [e for e in examples
              if e["metadata_case"] == "first_letter_large" and e["metadata_role"] in PRES
              and e["metadata_content_pres_kg"] > e["metadata_content_pres_gated"]]

for e in large_rows[:3]:
    print("PROMPT:", e["input"].split("]", 1)[-1].strip()[:90], "...")
    print(f"  NOOP  : {e['predict_noop'][:95]!s}")
    print(f"  KG-ABL  (f={e['metadata_fluency_kg']} c={e['metadata_content_pres_kg']} -> util {e['metadata_utility_kg']:.1f}): {e['predict_kg_abl'][:95]!s}")
    print(f"  GATED   (f={e['metadata_fluency_gated']} c={e['metadata_content_pres_gated']} -> util {e['metadata_utility_gated']:.1f}): {e['predict_dense_sub_gated'][:95]!s}")
    print("-" * 100)

## Aggregate verdict

The headline gate (verbatim from `method.py main()`): average the advantage over the **powered absorption**
cases that reach *meaningful forgetting* (`adv_absorption`) and over the **co-firing** cases
(`adv_cofiring`). The structural win requires `absorption_exceeds_cofiring` — i.e. the absorption advantage
must beat the co-firing one, so the win is *structure*, not merely edit footprint.

> **Mini-subset note.** This demo keeps 3 of the 5 cases (`large`, `insult`, `georgia`); `taxonomic_us`
> and `taxonomic_jordan` are omitted. So the demo's `adv_cofiring` (insult only) differs from the published
> 0.372 (mean of us + insult). The **overall verdict is unchanged**.

In [ ]:
cl        = list(cases.values())
abs_all   = [r for r in cl if r["regime"] == "absorption"]
abs_cases = [r for r in abs_all if not r["underpowered"]]      # powered absorption only
cof       = [r for r in cl if r["regime"] == "co-firing"]
beats     = [r for r in abs_cases if r["fork"] == F_BEATS]
abs_meaningful = [r for r in abs_cases if r["kg_can_forget"]]

def _adv(rs):
    vals = [r["adv"] for r in rs if r["adv"] is not None]
    return float(np.mean(vals)) if vals else None

adv_absorption = _adv(abs_meaningful)
adv_cofiring   = _adv(cof)
absorption_exceeds_cofiring = bool(adv_absorption is not None and adv_cofiring is not None
                                   and adv_absorption > adv_cofiring)

if len(beats) >= 1 and absorption_exceeds_cofiring:
    overall = "SPARSE_SAE_HANDLE_ESTABLISHED"
elif len(beats) == 0 and any(r["fork"] == F_CLOSES for r in abs_cases):
    overall = "DISCOVERY_IS_THE_VALUE_GATING_NOT_SAE_SPECIFIC"
elif len(beats) >= 1:
    overall = "SPARSE_SAE_HANDLE_ESTABLISHED"
else:
    overall = "SELECTIVE_LOW_COLLATERAL_PARTIAL_SUPPRESSION"

print(f"adv_absorption (meaningful) = {adv_absorption:+.4f}   over {[ 'first_letter_large' if r['absorber']==8463 else r['regime'] for r in abs_meaningful]}")
print(f"adv_cofiring                = {adv_cofiring:+.4f}")
print(f"absorption_exceeds_cofiring = {absorption_exceeds_cofiring}")
print(f"n_KG_BEATS_GATED (abs)      = {len(beats)}")
print(f"\nOVERALL VERDICT (recomputed) = {overall}")

pub = data["summary_published"]
print(f"OVERALL VERDICT (published)  = {pub['overall_verdict']}")
print("\nNOTE:", pub["note"])

## Results summary + visualization

A results table and a bar chart of the decisive advantage `KG-ABL − GATED-DENSE` per case, with 95 %
bootstrap CIs. Green = KG beats the footprint-matched gated dense; grey = no meaningful forgetting (the KG
handle is clean but *partial* — it barely edits, so it preserves trivially).

In [ ]:
# ---- text table ----
hdr = f"{'case':18s} {'regime':10s} {'adv(KG-GATED)':>14s} {'95% CI':>22s} {'excl0':>6s}  fork"
print(hdr); print("-" * len(hdr))
order = list(cases.keys())
for c in order:
    r = cases[c]; ci = r["ci"]
    short = {F_BEATS: "KG_BEATS", F_NOMEAN: "NO_MEANINGFUL", F_CLOSES: "GATED_CLOSES"}[r["fork"]]
    print(f"{c:18s} {r['regime']:10s} {r['adv']:>+14.3f} "
          f"[{ci['ci_lo']:+.3f},{ci['ci_hi']:+.3f}]".rjust(22) + f" {str(ci['excl_0']):>6s}  {short}")

# ---- bar chart ----
advs = [cases[c]["adv"] for c in order]
lo_err = [cases[c]["adv"] - cases[c]["ci"]["ci_lo"] for c in order]
hi_err = [cases[c]["ci"]["ci_hi"] - cases[c]["adv"] for c in order]
palette = {F_BEATS: "#2ca02c", F_CLOSES: "#ff7f0e", F_NOMEAN: "#9e9e9e"}
cols = [palette[cases[c]["fork"]] for c in order]

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.bar(range(len(order)), advs, yerr=[lo_err, hi_err], capsize=7, color=cols, edgecolor="black", zorder=3)
ax.axhline(0, color="black", lw=1.0, zorder=2)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([f"{c}\n({cases[c]['regime']})" for c in order], fontsize=9)
ax.set_ylabel("joint-utility advantage\nKG-ABL  -  GATED-DENSE")
ax.set_title("KG single-absorber vs footprint-matched gated-dense\n(95% paired-bootstrap CI; >0 favours KG)")
ax.grid(axis="y", ls=":", alpha=0.5, zorder=0)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=palette[F_BEATS],  label="KG beats gated dense"),
                   Patch(facecolor=palette[F_NOMEAN], label="No meaningful forget (partial)")],
          fontsize=8, loc="upper right")
plt.tight_layout(); plt.show()

print(f"\nOverall: {overall}  |  absorption_exceeds_cofiring={absorption_exceeds_cofiring} "
      f"({adv_absorption:+.2f} vs {adv_cofiring:+.2f})")